<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/01_Basic/L9_Model_Loading_and_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L9: Model Loading and Inference - From Hub to Predictions

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Basic  
**Lesson:** 9 of 15

---

## Learning Objectives

By the end of this lesson, you will:
- Understand the HuggingFace Hub and model repository structure
- Learn how to load pre-trained models for different tasks
- Master the pipeline API for quick inference
- Explore manual model loading and inference
- Work with BERT, GPT, and T5 model families
- Understand model configurations and tokenizer integration

---

## Concept: The HuggingFace Ecosystem

**HuggingFace Hub** is the central repository for pre-trained models, datasets, and ML applications.

### Why HuggingFace Hub?

**The Challenge:**
- Training models from scratch is expensive
- Need access to state-of-the-art models
- Want to share and collaborate on models

**The Solution:**
- Pre-trained models ready to use
- Standardized API for loading and inference
- Community-driven model sharing

### Key Components:

1. **Models Hub**
   - 100,000+ pre-trained models
   - Text, vision, audio, multimodal
   - Easy download and deployment

2. **Transformers Library**
   - Unified API for all models
   - AutoModel classes for automatic loading
   - Pipeline API for quick inference

3. **Model Cards**
   - Documentation for each model
   - Training details and limitations
   - Usage examples and citations

### Three Ways to Use Models:

1. **Pipeline API** - Simplest, high-level interface
2. **AutoModel Classes** - More control, standard interface
3. **Model-Specific Classes** - Full control, advanced usage

---

In [ ]:
# Step 1: Install and Import Required Libraries
!pip install transformers torch sentencepiece accelerate -q

import torch
from transformers import pipeline, AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM
from transformers import BertModel, GPT2LMHeadModel, T5ForConditionalGeneration
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Part 1: Pipeline API - The Easiest Way

**Pipelines** provide a high-level API for common NLP tasks. They handle tokenization, model inference, and post-processing automatically.

### Available Pipeline Tasks:

- **text-classification**: Sentiment analysis, topic classification
- **token-classification**: Named Entity Recognition (NER), POS tagging
- **question-answering**: Extract answers from context
- **text-generation**: Generate text continuations
- **summarization**: Summarize long texts
- **translation**: Translate between languages
- **fill-mask**: Fill in masked tokens (BERT-style)

### Pipeline Workflow:

```
Text Input -> Tokenization -> Model Inference -> Post-processing -> Output
```

All handled automatically!

---

In [ ]:
# Step 2: Using Pipeline API for Different Tasks

print("Pipeline API Examples\n")
print("=" * 60)

# 1. Sentiment Analysis
print("\n1. SENTIMENT ANALYSIS")
print("-" * 60)
sentiment_pipeline = pipeline("sentiment-analysis")

texts = [
    "I love this product! It's amazing!",
    "This is terrible. Very disappointed.",
    "It's okay, nothing special."
]

for text in texts:
    result = sentiment_pipeline(text)[0]
    print(f"Text: {text}")
    print(f"  Label: {result['label']}, Score: {result['score']:.4f}")
    print()

# 2. Named Entity Recognition
print("\n2. NAMED ENTITY RECOGNITION (NER)")
print("-" * 60)
ner_pipeline = pipeline("ner", grouped_entities=True)

text = "Apple Inc. was founded by Steve Jobs in Cupertino, California."
entities = ner_pipeline(text)

print(f"Text: {text}\n")
for entity in entities:
    print(f"  Entity: {entity['word']}")
    print(f"  Type: {entity['entity_group']}")
    print(f"  Score: {entity['score']:.4f}")
    print()

# 3. Question Answering
print("\n3. QUESTION ANSWERING")
print("-" * 60)
qa_pipeline = pipeline("question-answering")

context = """The transformer architecture was introduced in 2017 by Vaswani et al. 
It uses self-attention mechanisms to process sequences in parallel, making it much 
faster than recurrent neural networks. Transformers have become the foundation for 
modern language models like BERT, GPT, and T5."""

questions = [
    "When was the transformer architecture introduced?",
    "What mechanism does the transformer use?",
    "What are some models based on transformers?"
]

for question in questions:
    result = qa_pipeline(question=question, context=context)
    print(f"Q: {question}")
    print(f"A: {result['answer']} (score: {result['score']:.4f})")
    print()

print("\nPipelines make inference incredibly easy!")

## Part 2: Loading BERT Models

**BERT (Bidirectional Encoder Representations from Transformers)** is designed for understanding tasks.

### BERT Architecture:
- **Encoder-only** transformer
- **Bidirectional** context understanding
- **Pre-trained** on MLM and NSP tasks

### Common BERT Variants:
- **bert-base-uncased**: 110M parameters, lowercase
- **bert-large-uncased**: 340M parameters
- **distilbert-base-uncased**: 66M parameters, faster
- **roberta-base**: Optimized BERT variant

---

In [ ]:
# Step 3: Loading and Using BERT Models

print("Loading BERT Model\n")
print("=" * 60)

# Load BERT tokenizer and model
model_name = "bert-base-uncased"
print(f"Model: {model_name}\n")

tokenizer_bert = AutoTokenizer.from_pretrained(model_name)
model_bert = AutoModel.from_pretrained(model_name)

print(f"Model loaded successfully!")
print(f"Model type: {type(model_bert).__name__}")
print(f"Number of parameters: {model_bert.num_parameters():,}")
print(f"Vocabulary size: {tokenizer_bert.vocab_size:,}")
print()

# Perform inference
text = "The transformer architecture revolutionized natural language processing."

print(f"Input text: {text}\n")

# Tokenize
inputs = tokenizer_bert(text, return_tensors="pt")
print("Tokenized inputs:")
print(f"  Input IDs shape: {inputs['input_ids'].shape}")
print(f"  Tokens: {tokenizer_bert.convert_ids_to_tokens(inputs['input_ids'][0])}")
print()

# Get model outputs
with torch.no_grad():
    outputs = model_bert(**inputs)

print("Model outputs:")
print(f"  Last hidden state shape: {outputs.last_hidden_state.shape}")
print(f"  Pooler output shape: {outputs.pooler_output.shape}")
print()

print("Explanation:")
print("  - Last hidden state: [batch_size, sequence_length, hidden_size]")
print("  - Each token has a 768-dimensional embedding")
print("  - Pooler output: [CLS] token representation for classification")

## Part 3: Loading GPT Models

**GPT (Generative Pre-trained Transformer)** is designed for generation tasks.

### GPT Architecture:
- **Decoder-only** transformer
- **Unidirectional** (left-to-right) attention
- **Pre-trained** on causal language modeling

### Common GPT Variants:
- **gpt2**: 117M parameters
- **gpt2-medium**: 345M parameters
- **gpt2-large**: 774M parameters
- **gpt2-xl**: 1.5B parameters

---

In [ ]:
# Step 4: Loading and Using GPT Models

print("Loading GPT-2 Model\n")
print("=" * 60)

# Load GPT-2 tokenizer and model
model_name = "gpt2"
print(f"Model: {model_name}\n")

tokenizer_gpt = AutoTokenizer.from_pretrained(model_name)
model_gpt = AutoModelForCausalLM.from_pretrained(model_name)

print(f"Model loaded successfully!")
print(f"Model type: {type(model_gpt).__name__}")
print(f"Number of parameters: {model_gpt.num_parameters():,}")
print(f"Vocabulary size: {tokenizer_gpt.vocab_size:,}")
print()

# Generate text
prompt = "The future of artificial intelligence is"

print(f"Prompt: {prompt}\n")

# Tokenize
inputs = tokenizer_gpt(prompt, return_tensors="pt")
print(f"Input IDs shape: {inputs['input_ids'].shape}")
print()

# Generate
print("Generating text...\n")
with torch.no_grad():
    outputs = model_gpt.generate(
        inputs['input_ids'],
        max_length=50,
        num_return_sequences=3,
        temperature=0.8,
        do_sample=True,
        pad_token_id=tokenizer_gpt.eos_token_id
    )

print("Generated texts:\n")
for i, output in enumerate(outputs, 1):
    generated_text = tokenizer_gpt.decode(output, skip_special_tokens=True)
    print(f"{i}. {generated_text}")
    print()

print("GPT models excel at text generation!")

## Part 4: Loading T5 Models

**T5 (Text-to-Text Transfer Transformer)** treats all NLP tasks as text-to-text problems.

### T5 Architecture:
- **Encoder-decoder** transformer
- **Text-to-text** framework
- **Unified** approach to all tasks

### Common T5 Variants:
- **t5-small**: 60M parameters
- **t5-base**: 220M parameters
- **t5-large**: 770M parameters
- **flan-t5-base**: Instruction-tuned variant

---

In [ ]:
# Step 5: Loading and Using T5 Models

print("Loading T5 Model\n")
print("=" * 60)

# Load T5 tokenizer and model
model_name = "t5-small"
print(f"Model: {model_name}\n")

tokenizer_t5 = AutoTokenizer.from_pretrained(model_name)
model_t5 = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print(f"Model loaded successfully!")
print(f"Model type: {type(model_t5).__name__}")
print(f"Number of parameters: {model_t5.num_parameters():,}")
print(f"Vocabulary size: {tokenizer_t5.vocab_size:,}")
print()

# T5 uses task prefixes
tasks = [
    ("translate English to German: The house is wonderful.", "Translation"),
    ("summarize: The transformer architecture uses self-attention mechanisms to process sequences in parallel. It has become the foundation for modern NLP models.", "Summarization"),
    ("question: What is the capital of France? context: Paris is the capital and largest city of France.", "Question Answering")
]

print("T5 Multi-Task Examples\n")

for task_input, task_name in tasks:
    print(f"Task: {task_name}")
    print(f"Input: {task_input}\n")
    
    # Tokenize
    inputs = tokenizer_t5(task_input, return_tensors="pt")
    
    # Generate
    with torch.no_grad():
        outputs = model_t5.generate(
            inputs['input_ids'],
            max_length=50,
            num_beams=4,
            early_stopping=True
        )
    
    # Decode
    result = tokenizer_t5.decode(outputs[0], skip_special_tokens=True)
    print(f"Output: {result}")
    print("-" * 60)
    print()

print("T5 handles multiple tasks with a unified text-to-text approach!")

## Part 5: Model Configurations and Advanced Loading

Every model has a configuration that defines its architecture and behavior.

---

In [ ]:
# Step 6: Exploring Model Configurations

from transformers import AutoConfig

print("Model Configuration Examples\n")
print("=" * 60)

# Load configurations
models_to_explore = [
    "bert-base-uncased",
    "gpt2",
    "t5-small"
]

for model_name in models_to_explore:
    print(f"\n{model_name.upper()}")
    print("-" * 60)
    
    config = AutoConfig.from_pretrained(model_name)
    
    print(f"Architecture: {config.model_type}")
    print(f"Hidden size: {config.hidden_size}")
    print(f"Number of layers: {config.num_hidden_layers if hasattr(config, 'num_hidden_layers') else config.num_layers}")
    print(f"Number of attention heads: {config.num_attention_heads}")
    print(f"Vocabulary size: {config.vocab_size:,}")
    print(f"Max position embeddings: {config.max_position_embeddings if hasattr(config, 'max_position_embeddings') else 'N/A'}")
    
print("\n" + "=" * 60)
print("\nConfigurations define model architecture and behavior!")

## Part 6: Comparing Model Outputs

Let's compare how different models handle the same input.

---

In [ ]:
# Step 7: Model Comparison

import matplotlib.pyplot as plt
import numpy as np

print("Comparing Models on Same Task\n")
print("=" * 60)

# Task: Fill-mask (BERT) vs Text Generation (GPT-2)

# BERT Fill-Mask
print("\n1. BERT: Fill-Mask Task")
print("-" * 60)
fill_mask = pipeline("fill-mask", model="bert-base-uncased")

masked_text = "The transformer architecture is [MASK] for NLP tasks."
print(f"Input: {masked_text}\n")

results = fill_mask(masked_text)
print("Top 5 predictions:")
for i, result in enumerate(results, 1):
    print(f"  {i}. {result['token_str']:15s} (score: {result['score']:.4f})")

# GPT-2 Text Completion
print("\n2. GPT-2: Text Completion")
print("-" * 60)
text_gen = pipeline("text-generation", model="gpt2")

prompt = "The transformer architecture is"
print(f"Prompt: {prompt}\n")

results = text_gen(prompt, max_length=30, num_return_sequences=3, temperature=0.7)
print("Generated completions:")
for i, result in enumerate(results, 1):
    print(f"  {i}. {result['generated_text']}")

print("\n" + "=" * 60)
print("\nDifferent models excel at different tasks!")

## Part 7: Model Size and Performance Trade-offs

Let's analyze the trade-offs between model size and performance.

---

In [ ]:
# Step 8: Model Size Analysis

import time

print("Model Size and Performance Analysis\n")
print("=" * 60)

# Compare different BERT variants
bert_variants = [
    "distilbert-base-uncased",
    "bert-base-uncased",
    "bert-large-uncased"
]

model_stats = []

for model_name in bert_variants:
    print(f"\nLoading {model_name}...")
    
    # Load model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    
    # Get stats
    num_params = model.num_parameters()
    
    # Measure inference time
    text = "The transformer architecture revolutionized natural language processing."
    inputs = tokenizer(text, return_tensors="pt")
    
    start_time = time.time()
    with torch.no_grad():
        for _ in range(10):
            outputs = model(**inputs)
    inference_time = (time.time() - start_time) / 10
    
    model_stats.append({
        'name': model_name.split('-')[0],
        'params': num_params / 1e6,  # Convert to millions
        'time': inference_time * 1000  # Convert to milliseconds
    })
    
    print(f"  Parameters: {num_params:,} ({num_params/1e6:.1f}M)")
    print(f"  Inference time: {inference_time*1000:.2f}ms")

# Visualize comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

names = [s['name'] for s in model_stats]
params = [s['params'] for s in model_stats]
times = [s['time'] for s in model_stats]

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

# Parameters comparison
ax1.bar(names, params, color=colors, edgecolor='black', linewidth=2)
ax1.set_ylabel('Parameters (Millions)', fontsize=12)
ax1.set_title('Model Size Comparison', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

for i, (name, param) in enumerate(zip(names, params)):
    ax1.text(i, param, f'{param:.0f}M', ha='center', va='bottom', fontweight='bold')

# Inference time comparison
ax2.bar(names, times, color=colors, edgecolor='black', linewidth=2)
ax2.set_ylabel('Inference Time (ms)', fontsize=12)
ax2.set_title('Inference Speed Comparison', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

for i, (name, time_val) in enumerate(zip(names, times)):
    ax2.text(i, time_val, f'{time_val:.1f}ms', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("\nKey Insights:")
print("  - Larger models have more parameters and better performance")
print("  - Smaller models are faster and more efficient")
print("  - Choose based on your speed vs accuracy requirements")

## Part 8: Practical Tips for Model Loading

Best practices for working with models in production.

---

In [ ]:
# Step 9: Best Practices for Model Loading

print("Best Practices for Model Loading\n")
print("=" * 60)

# 1. Caching models locally
print("\n1. MODEL CACHING")
print("-" * 60)
print("Models are automatically cached in ~/.cache/huggingface/")
print("First load: Downloads from Hub")
print("Subsequent loads: Uses cached version")
print()

# 2. Loading specific model revisions
print("\n2. LOADING SPECIFIC REVISIONS")
print("-" * 60)
print("You can load specific model versions:")
print("  model = AutoModel.from_pretrained('bert-base-uncased', revision='main')")
print()

# 3. Device management
print("\n3. DEVICE MANAGEMENT")
print("-" * 60)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Current device: {device}")
print()
print("Move model to GPU:")
print("  model = model.to('cuda')")
print("Or specify during loading:")
print("  pipeline('sentiment-analysis', device=0)  # GPU 0")
print()

# 4. Memory optimization
print("\n4. MEMORY OPTIMIZATION")
print("-" * 60)
print("For large models, use:")
print("  - torch_dtype=torch.float16 for half precision")
print("  - device_map='auto' for automatic device placement")
print("  - load_in_8bit=True for 8-bit quantization")
print()
print("Example:")
print("  model = AutoModel.from_pretrained(")
print("      'bert-large-uncased',")
print("      torch_dtype=torch.float16,")
print("      device_map='auto'")
print("  )")
print()

# 5. Batch processing
print("\n5. BATCH PROCESSING")
print("-" * 60)
print("Process multiple inputs efficiently:")
print()

texts = [
    "This is great!",
    "This is terrible.",
    "This is okay."
]

sentiment = pipeline("sentiment-analysis")
results = sentiment(texts)

print("Batch inference results:")
for text, result in zip(texts, results):
    print(f"  {text:20s} -> {result['label']:8s} ({result['score']:.4f})")

print("\n" + "=" * 60)
print("\nFollowing best practices ensures efficient model usage!")

## Exercises

### Exercise 1: Explore Different Models
Load and compare different model variants:
```python
models = ['bert-base-uncased', 'roberta-base', 'albert-base-v2']
for model_name in models:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    # Compare vocabulary size, parameters, architecture
```

### Exercise 2: Multi-Task Pipeline
Create pipelines for different tasks and test them:
- Sentiment analysis on movie reviews
- NER on news articles
- Question answering on Wikipedia paragraphs
- Text generation with different prompts

### Exercise 3: Model Performance Benchmarking
Benchmark inference speed for different models:
```python
import time

text = "Your test text here"
for model_name in ['distilbert-base-uncased', 'bert-base-uncased']:
    model = AutoModel.from_pretrained(model_name)
    # Measure inference time over 100 runs
    # Compare speed vs model size
```

### Exercise 4: Custom Model Loading
Load models with custom configurations:
- Try different torch_dtype settings (float32, float16)
- Experiment with device placement
- Test batch processing with different batch sizes

### Exercise 5: Model Comparison Study
Compare BERT, GPT-2, and T5 on the same task:
- Choose a task (e.g., text completion)
- Adapt the task for each model's strengths
- Compare outputs and performance
- Document which model works best for what

---

## Key Takeaways

1. **HuggingFace Hub** provides access to 100,000+ pre-trained models
2. **Pipeline API** is the easiest way to use models for common tasks
3. **AutoModel classes** provide standardized loading for any model
4. **BERT** excels at understanding tasks (classification, NER, QA)
5. **GPT** excels at generation tasks (text completion, creative writing)
6. **T5** uses text-to-text framework for unified task handling
7. **Model configurations** define architecture and behavior
8. **Model size** involves trade-offs between performance and speed
9. **Best practices** include caching, device management, and batch processing
10. **Different models** are optimized for different tasks

### Model Selection Guide:

**Use BERT when:**
- Need bidirectional context understanding
- Working on classification or extraction tasks
- Want strong performance on understanding

**Use GPT when:**
- Need text generation capabilities
- Working on completion or creative tasks
- Want autoregressive generation

**Use T5 when:**
- Need unified approach to multiple tasks
- Working on seq2seq tasks (translation, summarization)
- Want flexibility in task formulation

---

## Additional Resources

### Documentation
- [HuggingFace Hub](https://huggingface.co/models)
- [Transformers Documentation](https://huggingface.co/docs/transformers/)
- [Pipeline API Guide](https://huggingface.co/docs/transformers/main_classes/pipelines)

### Papers
- **"BERT: Pre-training of Deep Bidirectional Transformers"** (Devlin et al., 2018)
- **"Language Models are Unsupervised Multitask Learners"** (Radford et al., 2019) - GPT-2
- **"Exploring the Limits of Transfer Learning with T5"** (Raffel et al., 2020)

### Tutorials
- [HuggingFace Course](https://huggingface.co/course/)
- [Model Hub Tour](https://huggingface.co/docs/hub/models)
- [Fine-tuning Tutorial](https://huggingface.co/docs/transformers/training)

### Interactive
- [Model Cards](https://huggingface.co/models) - Explore model documentation
- [Inference API](https://huggingface.co/inference-api) - Test models in browser

---

## Next Lesson

**L10: Text Generation Basics** - Learn about generation strategies: greedy search, beam search, sampling, temperature, top-k, and top-p

---

**Congratulations! You can now load and use pre-trained models!**

*You've learned how to access the vast ecosystem of pre-trained models and use them for various NLP tasks.*